# ResNet identity shortcut

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/03-identity/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本版使用 CPU 驗證機制及短步參數更新；長訓練的辨識效果、AP 與 GPU 效率仍待補測。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.1.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 本輪只驗證 CPU。已裝相同 PyTorch 版本時保留其 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Identity is an exact statement about same-shape addition."""
import torch
from torch import nn


class IdentityBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.branch = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1, bias=False),
                                    nn.ReLU(),
                                    nn.Conv2d(channels, channels, 3, padding=1, bias=False))

    def forward(self, x):
        return x + self.branch(x)  # No post-add ReLU in this teaching block.


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    block = IdentityBlock(1)
    with torch.no_grad():
        for p in block.parameters():
            p.zero_()
    x = torch.tensor([[[[-2.0, -1.0], [0.0, 1.0]]]], requires_grad=True)
    y = block(x)
    assert torch.equal(x, y)
    y.sum().backward()
    assert torch.equal(x.grad, torch.ones_like(x))
    print(f"x={x.detach().flatten().tolist()}, y={y.detach().flatten().tolist()}")
    print(f"input_gradient={x.grad.flatten().tolist()}; F=0 preserves negative values too")

    trained = IdentityBlock(4)
    inputs, target = torch.randn(2, 4, 8, 8), torch.zeros(2, 4, 8, 8)
    optimizer = torch.optim.SGD(trained.parameters(), lr=0.05)
    before = trained.branch[2].weight.detach().clone()
    trained.train()
    for step in range(2):
        optimizer.zero_grad(set_to_none=True)
        prediction = trained(inputs)
        loss = nn.functional.mse_loss(prediction, target)
        loss.backward()
        assert trained.branch[2].weight.grad.abs().sum() > 0
        optimizer.step()
        print(f"step={step}, shape={tuple(prediction.shape)}, loss={loss.item():.4f}")
    assert not torch.equal(before, trained.branch[2].weight)
    print("random branch updated; this does not measure ResNet classification quality")


if __name__ == "__main__":
    main()


x=[-2.0, -1.0, 0.0, 1.0], y=[-2.0, -1.0, 0.0, 1.0]
input_gradient=[1.0, 1.0, 1.0, 1.0]; F=0 preserves negative values too
step=0, shape=(2, 4, 8, 8), loss=1.0816
step=1, shape=(2, 4, 8, 8), loss=1.0724
random branch updated; this does not measure ResNet classification quality
